<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">


معیارهای مهم ارزیابی RAG
- Retrieval Precision	چه تعداد از نتایج بازیابی‌شده مرتبط هستند؟
- Retrieval Recall	چه تعداد از اسناد مرتبط موردنیاز پیدا شده‌اند؟
- Context Relevance	Context چقدر به سؤال کاربر مرتبط است؟
- Answer Relevance	پاسخ چقدر به سؤال کاربر مربوط است؟
- Faithfulness	آیا پاسخ با Context بازیابی‌شده سازگار است؟
- Groundedness	آیا ادعاهای پاسخ پشتوانه‌ای در منابع دارند؟

نکته مهم درباره کد:
- Precision و Recall در این مثال واقعاً محاسبه می‌شوند.
- مقادیر context_relevance، answer_relevance و faithfulness برای نمونه به‌صورت دستی برچسب‌گذاری شده‌اند؛ هنوز ارزیابی خودکار با LLM انجام نمی‌دهیم.
- در ارزیابی واقعی باید Dataset معتبر و Ground Truth مناسب داشته باشیم.
- این معیارها را نباید بدون توجه به تعریف و روش محاسبه‌شان با یکدیگر مقایسه کرد.

ابزارهای ارزیابی RAG
چند ابزار شناخته‌شده برای این کار وجود دارند:
- Ragas: مجموعه‌ای از معیارهای تخصصی برای ارزیابی RAG.
  - Ragas Documentation https://docs.ragas.io/en/stable/?utm_source=chatgpt.com
- LangSmith: ارزیابی و مقایسه عملکرد مراحل مختلف سیستم.
  - LangSmith Evaluation https://docs.langchain.com/langsmith/evaluation?utm_source=chatgpt.com
- Microsoft Foundry Evaluators: ارزیابی Retrieval، Groundedness و Relevance.
  - Official Documentation https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/rag-evaluators?utm_source=chatgpt.com

In [1]:
# 1. Evaluation Dataset
evaluation_data = [
    {
        "question": "How many annual leave days do employees receive?",
        "relevant_doc_ids": ["HR-001"],
        "retrieved_doc_ids": ["HR-001", "HR-003", "HR-004"],
        "retrieved_contexts": [
            "Employees receive 26 annual leave days per year.",
            "Employees have supplementary health insurance.",
            "Monthly salaries are paid at the end of each month."
        ],
        "answer": "Employees receive 26 annual leave days per year.",
        "reference_answer": "Employees receive 26 annual leave days per year.",
        "context_relevance": True,
        "answer_relevance": True,
        "faithfulness": True
    },
    {
        "question": "Who must approve annual leave requests?",
        "relevant_doc_ids": ["HR-002"],
        "retrieved_doc_ids": ["HR-001", "HR-003"],
        "retrieved_contexts": [
            "Employees receive 26 annual leave days per year.",
            "Employees have supplementary health insurance."
        ],
        "answer": "The HR department must approve annual leave requests.",
        "reference_answer": "Annual leave requests must be approved by the manager.",
        "context_relevance": False,
        "answer_relevance": True,
        "faithfulness": False
    }
]


# 2. Retrieval Metrics
def calculate_retrieval_metrics(expected_ids, retrieved_ids):

    expected = set(expected_ids)
    retrieved = set(retrieved_ids)

    relevant_retrieved = expected.intersection(retrieved)

    precision = (
        len(relevant_retrieved) / len(retrieved)
        if retrieved else 0
    )

    recall = (
        len(relevant_retrieved) / len(expected)
        if expected else 0
    )

    return precision, recall


# 3. Evaluate Each Sample
evaluation_results = []

for sample in evaluation_data:

    precision, recall = calculate_retrieval_metrics(
        sample["relevant_doc_ids"],
        sample["retrieved_doc_ids"]
    )

    evaluation_results.append({
        "question": sample["question"],
        "retrieval_precision": precision,
        "retrieval_recall": recall,
        "context_relevance": sample["context_relevance"],
        "answer_relevance": sample["answer_relevance"],
        "faithfulness": sample["faithfulness"]
    })


# 4. Display Results
for index, result in enumerate(evaluation_results, start=1):

    print(f"\nEvaluation Sample {index}")
    print("-" * 50)

    for metric, value in result.items():
        print(f"{metric}: {value}")


# 5. Calculate Average Metrics
metrics = [
    "retrieval_precision",
    "retrieval_recall",
    "context_relevance",
    "answer_relevance",
    "faithfulness"
]

print("\n" + "=" * 50)
print("AVERAGE EVALUATION SCORES")

for metric in metrics:

    average_score = sum(
        result[metric]
        for result in evaluation_results
    ) / len(evaluation_results)

    print(f"{metric}: {average_score:.2f}")


Evaluation Sample 1
--------------------------------------------------
question: How many annual leave days do employees receive?
retrieval_precision: 0.3333333333333333
retrieval_recall: 1.0
context_relevance: True
answer_relevance: True
faithfulness: True

Evaluation Sample 2
--------------------------------------------------
question: Who must approve annual leave requests?
retrieval_precision: 0.0
retrieval_recall: 0.0
context_relevance: False
answer_relevance: True
faithfulness: False

AVERAGE EVALUATION SCORES
retrieval_precision: 0.17
retrieval_recall: 0.50
context_relevance: 0.50
answer_relevance: 1.00
faithfulness: 0.50
